In [84]:
import os, sys
import h5py
import numpy as np
import pandas as pd
import cv2
import torch
import time
from matplotlib import pyplot as plt

## Experiment Parameters

In [85]:
is_local = True # todo

if is_local:
    sys.path.append(r'C:/Users/w37262do/Documents/git/IR_preprocessing')
else:
    sys.path.append(r'/mnt/iusers01/eee01/w37262do/scratch/IR_preprocessing')
from src.preprocessing_tk import *

# Preprocessing protocol
protocol_name = sys.argv[1] if not is_local else 'greedy'
protocol_args = sys.argv[2:] if not is_local else ['none','none','none','none','none',]

In [86]:
dset_name = "raw"

In [87]:
preprocessing_tk = get_processing_tk_from_name(protocol_name,protocol_args)

## Open raw dataset, sample data

In [88]:
save_base = r'G:\preprocessed_spectra_for_comparison' if is_local else r'./'
raw_wavenumbers = np.load(r'D:/datasets/qcl2026_raw/wavenumbers.npy' if is_local else r'./wavenumbers.npy')
loop_start_time = time.time()

In [89]:
print(f"Processing ...")
process_start_time = time.time()

# Import Data - AUS
h5_aus = h5py.File(
    r"F:/datasets/aus_qcl2026_raw/annotated_spectra_dataset.h5" if is_local else r'/mnt/eps01-rds/GardnerLab/aus_qcl2026_raw/annotated_spectra_dataset.h5'
)
spectra_aus = h5_aus['spectra'][:]
label_aus = h5_aus['tissue_class'][:]

# Import Data - NOCL
h5_nocl = h5py.File(
    r"D:/datasets/qcl2026_raw/annotated_spectra_dataset.h5" if is_local else r'/mnt/eps01-rds/GardnerLab/qcl2026_raw/annotated_spectra_dataset.h5'
)
spectra_nocl = h5_nocl['spectra'][:]
label_nocl = h5_nocl['tissue_class'][:]



Processing ...


In [90]:
take_per_class = 2000
idxs_nocl = []
idxs_aus = []

gen = np.random.default_rng(10447202)

for cls_idx in range(7):
    where_cls_nocl = np.where(label_nocl == cls_idx)
    idxs_nocl.append(gen.choice(where_cls_nocl[0], size=take_per_class, replace=False))

    where_cls_aus = np.where(label_aus == cls_idx)
    idxs_aus.append(gen.choice(where_cls_aus[0], size=take_per_class, replace=False))

idxs_nocl = np.concatenate(idxs_nocl)
spectra_nocl = spectra_nocl[idxs_nocl]

idxs_aus = np.concatenate(idxs_aus)
spectra_aus = spectra_aus[idxs_aus]

## Actually process and save the data

In [91]:
# Process Data
processed_nocl, processed_wavenumbers = preprocessing_tk.spectral_transform(spectra_nocl,raw_wavenumbers)
processed_aus, processed_wavenumbers = preprocessing_tk.spectral_transform(spectra_aus,raw_wavenumbers)

# Re-save data
hf = h5py.File(fr"{save_base}/{dset_name}/annotated_spectra_dataset_AUS.h5" , 'w')
hf.create_dataset('core_id', data=h5_aus['core_id'][:][idxs_aus].astype(int), compression=None)
hf.create_dataset('patient_id', data=h5_aus['patient_id'][:][idxs_aus].astype(int), compression=None)
hf.create_dataset('spectra', data=processed_aus.astype(np.float32), compression=None)
hf.create_dataset('tissue_class', data=h5_aus['tissue_class'][:][idxs_aus].astype(int), compression=None)
hf.close()
hf = h5py.File(fr"{save_base}/{dset_name}/annotated_spectra_dataset_NOCL.h5" , 'w')
hf.create_dataset('core_id', data=h5_nocl['core_id'][:][idxs_nocl].astype(int), compression=None)
hf.create_dataset('patient_id', data=h5_nocl['patient_id'][:][idxs_nocl].astype(int), compression=None)
hf.create_dataset('spectra', data=processed_nocl.astype(np.float32), compression=None)
hf.create_dataset('tissue_class', data=h5_nocl['tissue_class'][:][idxs_nocl].astype(int), compression=None)
hf.close()

np.save(fr"{save_base}/{dset_name}/wavenumbers.npy" ,processed_wavenumbers)



## End

In [92]:
# Finish
h5_aus.close()
h5_nocl.close()
process_time = time.time() - process_start_time
if not is_local: np.save(r'./wavenumbers.npy',processed_wavenumbers)
print(f"Finished processing all spectra!")
print(f"Processing completed in {process_time:.3F} seconds")

Finished processing all spectra!
Processing completed in 2.943 seconds
